# RetailMind AI — Day 4 draft

Reconstructed from the notebook state before the completed Day 4 workflow was written.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv('../data/Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.drop_duplicates(inplace=True)
df['Churn'] = df.loc[(df['Churn'] == 'Yes') | (df['Churn'] == 'No'), 'Churn']
df.dropna(subset=['Churn'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
df['tenure_bucket'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 72],
    labels=['0-12', '13-24', '25-48', '49-72']
)

df['charge_ratio'] = df['TotalCharges'] / (
    df['MonthlyCharges'] * df['tenure']
)

df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)
df['has_online_security'] = (df['OnlineSecurity'] == 'Yes').astype(int)

In [ ]:
x = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

xtrain, xtest, ytrain, ytest = train_test_split(
    x, y, test_size=0.2, shuffle=True, stratify=y, random_state=42
)

In [ ]:
cat_col = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'tenure_bucket'
]

num_col = [
    'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
    'charge_ratio', 'is_month_to_month', 'has_online_security'
]

In [ ]:
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encoding', OneHotEncoder(handle_unknown='ignore'))
])

pre = ColumnTransformer([
    ('num', num_pipe, num_col),
    ('cat', cat_pipe, cat_col)
])

In [ ]:
# Continue here: load the Day 3 baseline, then add feature selection and ensembles.
model = 